# Building AI Coding Agents with OpenClaw and MiniMax

**TechEx Hands-On Lab · 60 minutes · MiniMax × AI Valley**

> By minute 47, every attendee will have run their own AutoResearch loop and produced their own Elo curve. This notebook **is** the lab.

## How to run this notebook
1. **Run the next cell first** (the bootstrap). It clones the repo, installs `python-chess`, and wires your MiniMax key if you set one.
2. Then run each lab step in order. Each section has a **checkpoint** — pause there and pair with your neighbor.
3. Mock mode is the default — works without an API key. *Optional:* set `MINIMAX_API_KEY` in Colab Secrets (🔑 in the left sidebar) for live calls.
4. Stuck? Raise your hand — a TA will come over. You're already on the Colab fallback path, so most setup issues don't apply.

## Lab map
| Time | Block | What you do |
|---|---|---|
| 0–3 | The punchline | See the finished agent — your target |
| 3–8 | Architecture in one slide | OpenClaw four layers |
| 8–20 | **Lab Step 1–3** — Run the baseline | One iteration, read the trace · **Checkpoint 1** |
| 20–32 | **Lab Step 4** — Modify a tool | Change a description, see the schema diff · **Checkpoint 2** |
| 32–47 | **Lab Step 5** — AutoResearch on your laptop | 5-iteration loop, your own Elo curve · **Checkpoint 3** |
| 47–55 | Tiered extension | Pick beginner / intermediate / advanced |
| 55–58 | OpenClaw Gateway closer clip | The bridge to production |
| 58–60 | Resources & follow-up | Take-home links |

In [ ]:
"""Colab bootstrap — clone the repo, install deps, wire up MiniMax key,
snapshot the pristine bot files so each lab step can reset cleanly.

Idempotent — safe to re-run.
"""
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nickita-khylkouski/autoresearch-brief-challenge"
REPO_DIR = "autoresearch-brief-challenge"
# TODO: drop `-b feat/3-tool-calling-openclaw-agent` once that branch merges
# to main. Until then, the OpenClaw four-layer agent code lives only on
# that branch.
CLONE_BRANCH = "feat/3-tool-calling-openclaw-agent"

if IN_COLAB and not Path(REPO_DIR).exists():
    print(f"Cloning {REPO_URL} (branch: {CLONE_BRANCH}) ...")
    subprocess.run(["git", "clone", "--quiet", "-b", CLONE_BRANCH, REPO_URL], check=True)

if Path(REPO_DIR).exists():
    os.chdir(REPO_DIR)
elif not (Path("pyproject.toml").exists() and Path("autoresearch_chess").is_dir()):
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "autoresearch_chess").is_dir():
            os.chdir(candidate)
            break

print(f"Working directory: {Path.cwd()}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-chess"], check=True)
print("python-chess installed.")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Force bot/config.py to its pristine weak starting state. The current branch
# has a partially-improved config committed; we reset it so the AutoResearch
# loop has somewhere to climb.
PRISTINE_BOT_CONFIG = """SEARCH_DEPTH = 1
MATERIAL_WEIGHT = 1.0
MOBILITY_WEIGHT = 0.0
KING_SAFETY_WEIGHT = 0.0
USE_PIECE_SQUARES = False
CAPTURE_FIRST = True
"""

# Snapshot all four bot/* files in their pristine state. Each lab step can
# call reset_bot_to_baseline() to roll back any patches the agent committed.
Path("bot/config.py").write_text(PRISTINE_BOT_CONFIG, encoding="utf-8")
PRISTINE_BOT_FILES = {p.name: p.read_text(encoding="utf-8") for p in Path("bot").glob("*.py")}

def reset_bot_to_baseline():
    """Restore all bot/*.py files to their pristine snapshot."""
    for name, content in PRISTINE_BOT_FILES.items():
        Path(f"bot/{name}").write_text(content, encoding="utf-8")

print(f"Snapshotted {len(PRISTINE_BOT_FILES)} bot/*.py files at pristine baseline.")

# Wire up the MiniMax key from Colab Secrets (if any); otherwise force mock.
if IN_COLAB:
    try:
        from google.colab import userdata
        key = userdata.get("MINIMAX_API_KEY")
    except Exception:
        key = None
    if key:
        os.environ["MINIMAX_API_KEY"] = key
        os.environ.pop("AUTORESEARCH_MOCK_MINIMAX", None)
        print("MINIMAX_API_KEY loaded from Colab Secrets — live calls available.")
    else:
        os.environ["AUTORESEARCH_MOCK_MINIMAX"] = "1"
        print("Mock mode (no MINIMAX_API_KEY in Colab Secrets — full lab still works).")
elif not os.environ.get("MINIMAX_API_KEY"):
    os.environ["AUTORESEARCH_MOCK_MINIMAX"] = "1"
    print("Local mode without MINIMAX_API_KEY — mock mode.")

def use_mock():
    """Read the current mock-mode setting (toggled by the bootstrap above)."""
    return os.environ.get("AUTORESEARCH_MOCK_MINIMAX") == "1"

print(f"Python: {sys.version.split()[0]}")
print(f"Mock mode: {use_mock()}")

## §0 · 0–3 min · The Punchline (see what you'll build)

Before we explain anything, here's what every attendee in this room will produce by minute 47: an agent that improved a chess bot's Elo from ~630 to ~1280 across five iterations.

The image below is from a captured run. **You** will have a curve like this from your own laptop in 47 minutes.

In [ ]:
"""§0 — Display a captured Elo curve. This is your target."""
from pathlib import Path
from IPython.display import Image, display

replay_png = Path("artifacts/demo_replay/progress.png")
if replay_png.exists():
    display(Image(str(replay_png)))
else:
    print("(Captured progress.png not present — your own curve appears in §4.)")

## §1 · 3–8 min · Architecture in one slide

**Three ingredients of a modern coding agent:**

| Ingredient | Project | What it gives us |
|---|---|---|
| Architecture | **OpenClaw** | Four-layer pattern: Gateway · Context · ReAct · Tools |
| Model | **MiniMax** | A model that tool-calls reliably under JSON-schema |
| Loop | **AutoResearch** | Agent + objective eval + constrained surface + accept/reject |

**OpenClaw architecture ↔ four files in this repo:**

| OpenClaw layer | File | What the lab uses |
|---|---|---|
| Gateway (entry point) | `autoresearch_chess/agent/gateway.py` | `ChessAgent.run_iteration` |
| Context assembly | `autoresearch_chess/agent/context.py` | `build_initial_conversation` |
| ReAct runtime | `autoresearch_chess/agent/react.py` | `run_react_loop` |
| Tool layer | `autoresearch_chess/agent/tools.py` | `TOOLS` (5 tools) |

**One sentence on MiniMax:** the loop only compounds if the model can tool-call reliably under JSON-schema — you'll feel why in 25 minutes.

**Pair with your neighbor now.** At 80 attendees, two-person debugging scales 2× with zero overhead.

In [ ]:
"""§1 — Architecture in code. Each OpenClaw layer at a glance."""
import inspect
from autoresearch_chess.agent.gateway import ChessAgent
from autoresearch_chess.agent.context import build_initial_conversation
from autoresearch_chess.agent.react import run_react_loop
from autoresearch_chess.agent.tools import TOOLS

print("=== Tool layer — 5 tools ===")
for t in TOOLS:
    marker = "🛑 terminal" if t.terminal else "📖 read-only"
    print(f"  {marker}  {t.name}")
    print(f"             {t.description}")

print("\n=== Context layer ===")
print(f"  {inspect.signature(build_initial_conversation)}")

print("\n=== ReAct layer ===")
print(f"  {inspect.signature(run_react_loop)}")

print("\n=== Gateway layer ===")
print(f"  class ChessAgent — entry point that wires the three above together")

print("\nTo see a full layer's source, run:")
print("  print(inspect.getsource(ChessAgent))")
print("Or open the file from Colab's left sidebar (folder icon).")

## §2 · 8–20 min · Lab Step 1–3 — Run the baseline

**Goal:** every attendee runs **one** full agent iteration end-to-end on their own machine. When you're done you'll have a tool-call trace from the agent on disk and you'll have read it.

What happens when you run the next cell:
1. The agent sees the baseline Elo + the list of editable files
2. It calls tools to inspect the bot's source (`list_bot_files`, `read_bot_file`)
3. It proposes a unified diff via the terminal `propose_patch` tool
4. The eval runs and scores the patched bot
5. **Accept** (Elo improvement above threshold) or **reject** (commit/discard the patch)

Takes ~30–60 seconds in mock mode.

In [ ]:
"""Lab Step 1-3 — Run one full agent iteration end-to-end."""
from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG

reset_bot_to_baseline()
config = replace(STAGE_LOOP_CONFIG, iterations=1, mock_minimax=use_mock())
print(f"Running one iteration (mock={use_mock()})...\n")
summary_baseline = run_loop(config)

print(f"\n=== ITERATION COMPLETE ===")
print(f"Run dir:        {summary_baseline['run_dir']}")
print(f"Decision:       {'✅ ACCEPTED' if summary_baseline['accepted'] else '❌ rejected'}")
print(f"Baseline Elo:   {summary_baseline['baseline_eval'].get('estimated_elo'):.1f}")
print(f"Best Elo:       {summary_baseline['best_eval'].get('estimated_elo'):.1f}")

In [ ]:
"""Lab Step 1-3 — Read your agent's tool-call trace.

`agent_trace.jsonl` has one event per ReAct round. This is the agent's
reasoning, made inspectable.

If the agent iteration errored out before a single ReAct round completed
(e.g. live MiniMax auth/network failure), the trace file won't exist.
We surface the error from decision.json instead so you can diagnose.
"""
import json
from pathlib import Path

run_dir = Path(summary_baseline["run_dir"])
iter_dir = run_dir / "iterations/001"
trace_path = iter_dir / "agent_trace.jsonl"

if trace_path.exists():
    for line in trace_path.read_text(encoding="utf-8").splitlines():
        event = json.loads(line)
        calls = ", ".join(tc["name"] for tc in event["tool_calls"]) or "<final message>"
        print(f"\nround {event['round']}: {calls}")
        for tc in event["tool_calls"]:
            preview = tc["result_preview"].replace("\n", " ")[:140]
            print(f"  args:    {tc['arguments']}")
            print(f"  result:  {preview}...")
    print(f"\n\nFull trace on disk: {trace_path}")
    print("Open it via Colab's file browser (folder icon, left sidebar).")
else:
    # No trace = the iteration errored before any ReAct round completed.
    print("⚠️  No agent_trace.jsonl was written — the iteration failed before any")
    print("ReAct round completed. The reason is in decision.json:\n")
    decision_path = iter_dir / "decision.json"
    if decision_path.exists():
        decision = json.loads(decision_path.read_text(encoding="utf-8"))
        for reason in decision.get("reasons", []):
            print(f"  • {reason}")
        print()
        if not use_mock():
            print("You're in LIVE mode. Most common causes for a first-call failure:")
            print("  - MINIMAX_API_KEY invalid or expired")
            print("  - MiniMax API host unreachable from Colab")
            print("  - Wrong API mode (try setting MINIMAX_API_MODE=openai in Colab Secrets)")
            print()
            print("Quickest workaround: clear MINIMAX_API_KEY from Colab Secrets, re-run")
            print("the bootstrap cell (mock mode kicks in), then re-run Lab Step 1-3.")
        else:
            print("You're in mock mode — this shouldn't normally happen. Re-run bootstrap.")
    else:
        print(f"  (decision.json missing — check {iter_dir})")

### ✋ Checkpoint 1 — ~minute 18

**Thumbs up if your trace printed above** (you should see `round 1: list_bot_files`, `round 2: read_bot_file`, `round 3: propose_patch`).

**Stuck?** Most common causes:
- Bootstrap cell didn't complete → re-run it
- `ModuleNotFoundError: No module named 'autoresearch_chess'` → bootstrap didn't finish; re-run it
- Cell hung → click ⏹ then re-run

**If still stuck after 2 minutes, raise your hand — a TA will come over.** You're already in Colab, so local-environment issues don't apply.

## §3 · 20–32 min · Lab Step 4 — Modify a tool, see the schema change

**Goal:** see firsthand that tool descriptions shape the JSON-schema the model sees.

The form in the next cell picks one tool and replaces its description **in memory** (no file editing needed). The cell prints the before/after JSON — that's what MiniMax actually receives.

**In live mode** (MINIMAX_API_KEY set), the model will read your new description and may choose tools differently. **In mock mode**, the trace is scripted so the agent's choices stay the same — but the JSON-schema definitely changed, and that's the lesson: tool descriptions are how you steer a tool-calling model.

In [ ]:
"""Lab Step 4 — Modify one tool's description in memory.

Pick a tool from the dropdown and type a new description. The change applies
to TOOLS in this notebook session only (restart kernel to undo).
"""
import json
from dataclasses import replace as dc_replace
from autoresearch_chess.agent import tools as tools_module

tool_to_modify = "propose_patch"  # @param ["list_bot_files", "read_bot_file", "get_baseline_eval", "get_recent_history", "propose_patch"]
new_description = "Does something to a file. Use this when you need to."  # @param {type:"string"}

original = next(t for t in tools_module.TOOLS if t.name == tool_to_modify)
modified = dc_replace(original, description=new_description)
idx = tools_module.TOOLS.index(original)
tools_module.TOOLS[idx] = modified
tools_module.TOOLS_BY_NAME[tool_to_modify] = modified

print(f"=== BEFORE ===")
print(json.dumps(original.to_openai_format(), indent=2))
print(f"\n=== AFTER ===")
print(json.dumps(modified.to_openai_format(), indent=2))
print("\nThe second JSON above is what MiniMax now receives as the tool definition.")

In [ ]:
"""Lab Step 4 — Re-run an iteration with the modified tool.

In LIVE mode, the model will read your new description and may behave differently.
In MOCK mode, the trace is scripted — so you won't see a behavior difference,
but you can verify the modified schema was sent.
"""
from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG

reset_bot_to_baseline()
config = replace(STAGE_LOOP_CONFIG, iterations=1, mock_minimax=use_mock())
summary_modified = run_loop(config)

print(f"\nRun dir: {summary_modified['run_dir']}")
if use_mock():
    print("(Mock mode — trace is scripted; the schema change above is the result.)")
else:
    print("(Live mode — re-read iterations/001/agent_trace.jsonl above to spot any behavior change.)")

### ✋ Checkpoint 2 — ~minute 30

**Thumbs up if you modified a tool description and saw the schema diff above.**

**Bonus thumbs up if you have `MINIMAX_API_KEY` set and re-ran the iteration in live mode** — did the agent's choices change?

This is where the MiniMax tool-calling story lands experientially: tool descriptions are part of the model's input, and good ones matter. No promotional framing required.

## §4 · 32–47 min · Lab Step 5 — AutoResearch on YOUR laptop (the headline)

This is the headline. Every attendee runs the same loop on their own machine and produces their own Elo curve. 80 curves climbing in parallel.

**AutoResearch = agent + objective eval + constrained edit surface + accept/reject.**

While the cell runs (~3–5 minutes), the principles:
- The eval is the only thing the agent cannot fake
- Bad ideas are cheap; good ideas compound
- This is a miniature of how MiniMax-style self-improvement loops work in production

The presenter's machine is running the same loop on the projector — a **leader curve** you can use as a sync reference.

In [ ]:
"""Lab Step 5 — Your AutoResearch loop. ~3-5 minutes."""
iterations = 5  # @param {type:"slider", min:1, max:10, step:1}

from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG

reset_bot_to_baseline()
print(f"Resetting bot to pristine baseline; running {iterations} iterations (mock={use_mock()})...\n")

config = replace(STAGE_LOOP_CONFIG, iterations=iterations, mock_minimax=use_mock())
summary = run_loop(config)

print(f"\n=== YOUR RUN ===")
print(f"Run ID:        {summary['run_id']}")
print(f"Baseline Elo:  {summary['baseline_eval'].get('estimated_elo'):.1f}")
print(f"Best Elo:      {summary['best_eval'].get('estimated_elo'):.1f}")
print(f"Accepted:      {summary['accepted']} / {summary['accepted'] + summary['rejected']}")
print(f"Run dir:       {summary['run_dir']}")

In [ ]:
"""Your Elo curve — the headline visual."""
from pathlib import Path
from IPython.display import Image, display

run_dir = Path(summary["run_dir"])
display(Image(str(run_dir / "progress.png")))

In [ ]:
"""Lab Step 5 — Inspect one accepted and one rejected patch."""
import json
from pathlib import Path

run_dir = Path(summary["run_dir"])
events = [
    json.loads(line)
    for line in (run_dir / "progress.jsonl").read_text(encoding="utf-8").splitlines()
]
iter_events = [e for e in events if e.get("event") == "iteration"]
first_accept = next((e for e in iter_events if e["decision"] == "accepted"), None)
first_reject = next((e for e in iter_events if e["decision"] == "rejected"), None)

def show(label, event):
    if event is None:
        print(f"=== {label}: none in this run ===\n")
        return
    n = event["iteration"]
    it_dir = run_dir / "iterations" / f"{n:03d}"
    decision = json.loads((it_dir / "decision.json").read_text(encoding="utf-8"))
    print(f"=== {label} · iteration {n} ===")
    print(f"candidate Elo: {event['candidate_elo']}    best Elo: {event['best_elo']}")
    print(f"reasons:       {decision.get('reasons', [])}")
    print(f"improvement:   {decision.get('improvement', 'n/a')}")
    print(f"--- patch ({it_dir / 'patch.diff'}) ---")
    diff_text = (it_dir / "patch.diff").read_text(encoding="utf-8")
    print("\n".join(diff_text.splitlines()[:30]))
    print()

show("ACCEPTED", first_accept)
show("REJECTED", first_reject)

### ✋ Checkpoint 3 — ~minute 46

**Hands up if your Elo went up. Hands up if it went down.** Both are valid — both teach.

The variance across the room IS the lesson:
- Mock mode is deterministic — everyone with the same iteration count gets the same curve
- Live mode produces different patches → different curves → different Elos
- That's AutoResearch in action: same architecture, real model variance, eval is the truth

**Compare with your neighbor.** If both of you ran mock, your curves match exactly. If one of you ran live, they diverge — that's a feature, not a bug.

## §5 · 47–55 min · Tiered Extensions — pick ONE

Three options, three difficulty levels. Pick the cell that matches your comfort. TAs are roaming to help.

- 🟢 **Beginner** — Customize the system prompt
- 🟡 **Intermediate** — Add a 5th tool (`list_files`) the agent can call
- 🔴 **Advanced** — Bias the agent's heuristic, compare Elo trajectories

### 🟢 Beginner extension — Customize the system prompt

The system prompt is what tells the agent its goal and the constraints. Change it to something different — for example, ask the agent to *also* explain its patches in plain English before submitting. Re-run the 5-iteration cell after.

In [ ]:
"""🟢 Beginner — Edit the agent's system prompt."""
custom_system_prompt = """You are MiniMax operating inside an AutoResearch chess loop.

Your job: improve the chess bot's estimated Elo by emitting one unified diff patch.

BEFORE you propose a patch, write 2-3 sentences explaining in plain English what
heuristic the patch encodes and why you think it will help.

You may edit only: {editable}.
Do not edit: {forbidden_files}.

Baseline Elo: {baseline_elo}. Current best: {best_elo}.
Only patches that improve estimated Elo are accepted.

Use list_bot_files, read_bot_file, and get_recent_history to inspect the code,
then propose_patch with a small targeted diff."""  # @param {type:"string"}

from autoresearch_chess.agent import context as ctx_module
ctx_module.SYSTEM_PROMPT = custom_system_prompt
print("Custom prompt installed. Scroll up to Lab Step 5 and re-run the 5-iteration cell.")
print("\nNote: in mock mode, the trace stays scripted regardless of the prompt — switch")
print("to live mode (MINIMAX_API_KEY in Colab Secrets) to see your prompt actually take effect.")

### 🟡 Intermediate extension — Add a 5th tool: `list_files`

The agent today has `list_bot_files`, which only returns the 4 editable files. Add a new read-only `list_files(path)` tool that lists files under any directory in the repo — gives the agent broader visibility (it still can't *edit* anything outside the editable surface; the constraint is enforced by `propose_patch`).

The skeleton, schema, and handler are pre-written below. Once it works, try modifying the handler to filter by extension, or write a totally different tool like `grep_bot_files`.

In [ ]:
"""🟡 Intermediate — Add a new `list_files` tool to the agent's toolkit."""
import json
from autoresearch_chess.agent.tools import Tool, TOOLS, TOOLS_BY_NAME

def list_files_handler(args, ctx):
    """Return entries (files and subdirs) under a relative path. Read-only."""
    path = str(args.get("path", ".")).strip() or "."
    target = ctx.root / path
    if not target.exists() or not target.is_dir():
        return json.dumps({"error": f"not_a_directory:{path}"})
    # Skip hidden dirs and noisy build artifacts
    entries = sorted(
        p.name + ("/" if p.is_dir() else "")
        for p in target.iterdir()
        if not p.name.startswith(".") and p.name not in ("__pycache__", "node_modules")
    )
    # Cap the listing so the agent isn't overwhelmed
    return json.dumps({"path": path, "entries": entries[:50]})

new_tool = Tool(
    name="list_files",
    description=(
        "List the files and subdirectories under a path relative to the repo root. "
        "Read-only. Use this to discover what code exists; you can still only edit "
        "the files returned by list_bot_files."
    ),
    schema={
        "type": "object",
        "properties": {
            "path": {
                "type": "string",
                "description": "Relative path from the repo root. Default: '.' (repo root).",
            }
        },
        "additionalProperties": False,
    },
    handler=list_files_handler,
)

if not any(t.name == new_tool.name for t in TOOLS):
    TOOLS.append(new_tool)
    TOOLS_BY_NAME[new_tool.name] = new_tool
    print(f"✅ Added '{new_tool.name}'. The agent now has {len(TOOLS)} tools.")
else:
    print(f"'{new_tool.name}' already added. {len(TOOLS)} tools registered.")

print(f"\nVerify the new tool is in the schema MiniMax sees:")
from autoresearch_chess.agent.tools import openai_tool_payload
print(f"  Tool names: {[t['function']['name'] for t in openai_tool_payload()]}")

print("\nIn LIVE mode (MINIMAX_API_KEY set), re-run Lab Step 5 — the agent will see")
print("the new tool in its schema and may decide to call it. In mock mode, the agent")
print("follows its scripted flow regardless.")

### 🔴 Advanced extension — Bias the agent's heuristic, compare trajectories

The agenda calls for biasing the agent toward a specific chess heuristic and comparing the resulting Elo trajectory against the unbiased run from §4.

In the legacy single-shot mode the agent uses `autoresearch_chess/prompt_builder.py`. In the tool-calling mode you're running today, the equivalent lever is the **system prompt** in `autoresearch_chess/agent/context.py` plus the **initial user message** that kicks off each iteration.

The next cell patches `build_initial_conversation` to append your chosen heuristic bias to the user message. Each iteration of the loop will see your guidance. Then run 5 iterations and compare the Elo curve against your §4 run (held in `summary`).

In [ ]:
"""🔴 Advanced — Bias the agent toward a chess heuristic and re-run.

Pick a heuristic bias in the form below. The cell monkey-patches the gateway's
view of `build_initial_conversation` so every iteration of the loop sees your
guidance appended to the kickoff user message. Then runs 5 iterations and
compares the resulting best Elo against your §4 baseline (in `summary`).
"""
heuristic_bias = "Focus your patches on endgame heuristics: king activity in the endgame, passed-pawn evaluation, and opposition. Avoid changes that primarily help opening play."  # @param {type:"string"}

from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG
from autoresearch_chess.agent import context as ctx_module
from autoresearch_chess.agent import gateway as gw_module

# Wrap the real build_initial_conversation so the bias lands in the user message
# of every iteration. The gateway looks up `build_initial_conversation` in its
# own module namespace at call time, so patching `gw_module` is what matters.
_original_build = ctx_module.build_initial_conversation

def _biased_build(*, baseline_eval, best_eval):
    convo = _original_build(baseline_eval=baseline_eval, best_eval=best_eval)
    convo.messages[-1]["content"] += f"\n\nADDITIONAL HEURISTIC GUIDANCE: {heuristic_bias}"
    return convo

gw_module.build_initial_conversation = _biased_build

reset_bot_to_baseline()
config = replace(STAGE_LOOP_CONFIG, iterations=5, mock_minimax=use_mock())
print(f"Running 5 iterations with heuristic bias:\n  {heuristic_bias!r}\n")
summary_biased = run_loop(config)

# Restore the unbiased build so re-running other cells is unaffected
gw_module.build_initial_conversation = _original_build

# Compare
print(f"\n=== TRAJECTORY COMPARISON ===")
unbiased_best = summary["best_eval"].get("estimated_elo") if "summary" in dir() else None
if unbiased_best is not None:
    print(f"  §4 unbiased run:  best Elo = {unbiased_best:.1f}")
print(f"  Biased run:       best Elo = {summary_biased['best_eval'].get('estimated_elo'):.1f}")
print(f"                    accepted = {summary_biased['accepted']} / {summary_biased['accepted'] + summary_biased['rejected']}")
print(f"\nIn LIVE mode, the bias actually changes what the agent proposes — different")
print(f"biases produce different Elo trajectories. Try several biases (e.g. 'aggressive")
print(f"king attack', 'positional/quiet play') and watch the curves diverge.")
print(f"\nIn MOCK mode, the agent's patches are scripted so the trajectory matches §4")
print(f"regardless of bias — the bias text is in the prompt but doesn't change behavior.")

## §6 · 55–58 min · The Closer — OpenClaw Gateway clip

> *"What you ran on your laptop today was the OpenClaw architecture in-process. Here's the same agent on the actual OpenClaw Gateway. The migration is mechanical — see [`docs/openclaw_mapping.md`](./docs/openclaw_mapping.md). This is what production looks like, three days of work from where you are now."*

*(30–60 second recorded clip plays here — the agent running through the actual OpenClaw Gateway with the same five tools.)*

The architectural seams you saw today (Tool / Context / ReAct / Gateway as separate layers) are exactly what makes Gateway deployment mechanical. You taught yourself the shape; the migration is the rest of the work.

## §7 · 58–60 min · Resources & follow-up

Replacing open Q&A — at 80 people, open Q&A gets three loud voices and 77 silent ones. The questions board you've been adding to during the lab is what the presenter answers here. Everything else gets a follow-up in the channel.

### Take this home

- 📂 **Repo:** [`nickita-khylkouski/autoresearch-brief-challenge`](https://github.com/nickita-khylkouski/autoresearch-brief-challenge) — `git clone` for a real dev environment after the event
- 📖 **OpenClaw migration guide:** `docs/openclaw_mapping.md` in the cloned repo
- 💬 **Follow-up channel:** *[link added in setup email]*
- 🎁 **MiniMax credits:** *[link added in setup email]*
- ❓ **Questions board:** *[Slido / shared doc link]*

### Your run artifacts (download below)

Colab's filesystem is ephemeral — when the runtime disconnects, everything you produced disappears. The next cell zips your AutoResearch run (trace, patches, Elo curve) into a downloadable archive.

In [ ]:
"""Hand-off artifact: download your run."""
import shutil
import sys
from pathlib import Path

run_dir = Path(summary["run_dir"])
archive_base = Path.cwd() / f"workshop_run_{summary['run_id']}"
archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(run_dir.parent),
    base_dir=run_dir.name,
)
print(f"Archive: {archive_path}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive_path)
else:
    print("(Not in Colab — archive saved to the path above; no browser download triggered.)")

---

### 🛠 Stage-reliability fallback — Replay a captured run

**Skip this cell unless the live loop failed.** Drops in a captured successful run from `artifacts/demo_replay/` so the Elo-curve narrative survives even if your kernel crashed mid-loop or MiniMax went down.

In [ ]:
import sys
!{sys.executable} -m autoresearch_chess.replay --run artifacts/demo_replay